## Federated Learning Lab: Logistic regression for classification of hand-drawn gestures


For this lab, you will train a logistic regression classifier to recognize hand-drawn gestures from the Quick Draw dataset. Then you will test it on a photo of YOUR own hand drawing.

After training your own model, you will compare it with a model trained using Federated Learning (FL), where you and your partner collaboratively train a model without sharing your private data.


#### Train your classifier

First, we'll train a logistic regression for classification of hand-drawn gesture images, using a subset of the Quick Draw dataset. 200 samples per class for your student slice.

(The instructor prepared the data files in advance using the classroom server.)


In [ ]:
!pip install flwr scikit-learn matplotlib numpy Pillow nest-asyncio seaborn --quiet


In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import urllib.request
import warnings
from PIL import Image
import PIL.ImageOps
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
import flwr as fl

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100

print('All packages imported.')


In [ ]:
STUDENT_ID = 'A'                # 'A' or 'B'
SERVER_ADDRESS = 'localhost:8080'

CLASSES = ['cat', 'dog', 'sun', 'clock', 'mountain',
           'tent', 'tree', 'bird', 'star', 'face']
nclasses = len(CLASSES)
SERVER_IP = SERVER_ADDRESS.split(':')[0]
HTTP_PORT = 8081

print(f'Student: {STUDENT_ID}  |  Server: {SERVER_ADDRESS}')


In [ ]:
# Load Quick Draw data for this student (like fetch_openml)
# The pre-built .npy file has 784 pixel columns + 1 label column
url = f'http://{SERVER_IP}:{HTTP_PORT}/student_{STUDENT_ID}_data.npy'
urllib.request.urlretrieve(url, 'student_data.npy')
data = np.load('student_data.npy')
X, y = data[:, :-1], data[:, -1].astype(int)
print(f'Loaded {len(X)} samples ({len(X)//nclasses} per class)')


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=9,
                                     train_size=0.7, test_size=0.3)


In [ ]:
X_train_scaled = X_train / 255.0
X_test_scaled = X_test / 255.0


In [ ]:
clf = LogisticRegression(penalty='l2',
                         max_iter=1000, solver='saga').fit(X_train_scaled, y_train)


In [ ]:
accuracy = clf.score(X_test_scaled, y_test)


In [ ]:
print(accuracy)


#### Create a test image

On a plain white piece of paper, in a black or other dark-colored pen, draw **one** of the 10 gestures (cat, dog, sun, clock, mountain, tent, tree, bird, star, or face). Take a photo of your hand drawing.

Edit your photo (crop, rotate as needed) so that it is approximately square, and includes only the drawing and the white background. Leave a small margin around the edge of the drawing, but not too much.


#### Upload your image to Colab

Run the following cell. Click "Choose files", and upload the photo of your hand drawing.


In [ ]:
from google.colab import files
 
uploaded = files.upload()
 
for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))


#### Visualize the image

After uploading, run this cell, but **replace the filename** with the name of the file you uploaded. You should see your image appear.


In [ ]:
from PIL import Image
 
filename = 'YOUR_FILENAME_HERE'
 
image = Image.open(filename)
p = plt.imshow(np.asarray(image), cmap=plt.cm.gray,);
p = plt.title('Shape: ' + str(np.asarray(image).shape))


#### Pre-process the image

The images in Quick Draw have been pre-processed — they are converted to grayscale, resized to 28x28, and inverted (white strokes on black background).

Use the code in the following cells to pre-process your image into a 28x28 image with one color channel (grayscale). You may have to manually tune the contrast for best results, by changing the `pixel_filter` value.


In [ ]:
# convert to grayscale - 'L' format = one value per pixel (0-255)
image_bw = image.convert('L')
p = plt.imshow(np.asarray(image_bw), cmap=plt.cm.gray,);
p = plt.title('Shape: ' + str(np.asarray(image_bw).shape))


In [ ]:
# resize to 28x28 (same as Quick Draw)
image_bw_resized = image_bw.resize((28, 28), Image.LANCZOS)
p = plt.imshow(np.asarray(image_bw_resized), cmap=plt.cm.gray,);
p = plt.title('Shape: ' + str(np.asarray(image_bw_resized).shape))


In [ ]:
# invert image, to match training data (white strokes on black background)
image_bw_resized_inverted = PIL.ImageOps.invert(image_bw_resized)
p = plt.imshow(np.asarray(image_bw_resized_inverted), cmap=plt.cm.gray,);
p = plt.title('Shape: ' + str(np.asarray(image_bw_resized_inverted).shape))


In [ ]:
# adjust contrast: clip low values, scale to [0, 1]
pixel_filter = 20  # may need to adjust (0-100)
min_pixel = np.percentile(image_bw_resized_inverted, pixel_filter)
image_scaled = np.clip(image_bw_resized_inverted - min_pixel, 0, 255)
max_pixel = np.max(image_scaled)
image_scaled = np.asarray(image_scaled) / max_pixel
p = plt.imshow(np.asarray(image_scaled), cmap=plt.cm.gray,);
p = plt.title('Shape: ' + str(np.asarray(image_scaled).shape))


In [ ]:
# reshape to (1, 784) - 1 sample, 784 features
test_sample = np.array(image_scaled).reshape(1, 784)
p = plt.imshow(np.reshape(test_sample, (28, 28)), cmap=plt.cm.gray,);
p = plt.title('Shape: ' + str(test_sample.shape))


#### Visualize the pre-processed image

Run the following code to view your final pre-processed image.


In [ ]:
p = plt.imshow(np.reshape(test_sample, (28, 28)), cmap=plt.cm.gray,);
p = plt.title('Shape: ' + str(test_sample.shape))


#### Use your fitted logistic regression

Now that you have processed your test image, let us see whether it is classified correctly by the logistic regression.


In [ ]:
test_probs = clf.predict_proba(test_sample)

sns.barplot(x=CLASSES, y=test_probs.squeeze());
plt.ylabel('Probability');
plt.xlabel('Class');


In [ ]:
test_pred = clf.predict(test_sample)
print('Predicted class is:', CLASSES[test_pred[0]])


#### Explain the model prediction

Even if the fitted model correctly labeled your drawing, it may have estimated a moderately high probability for some of the other labels. To understand why, it is useful to visualize the coefficient vector for each class.

Run the cell below. It plots:
- on the **top row**, the coefficient vector for each class,
- on the **bottom row**, each pixel in your test image, multiplied by the associated coefficient for that class.


In [ ]:
scale = np.max(np.abs(clf.coef_))

p = plt.figure(figsize=(25, 5));

for i in range(nclasses):
    p = plt.subplot(2, nclasses, i + 1)
    p = plt.imshow(clf.coef_[i].reshape(28, 28),
                  cmap=plt.cm.RdBu, vmin=-scale, vmax=scale);
    p = plt.title(CLASSES[i]);
    p = plt.axis('off')

for i in range(nclasses):
    p = plt.subplot(2, nclasses, nclasses + i + 1)
    p = plt.imshow(test_sample.reshape(28, 28) * clf.coef_[i].reshape(28, 28),
                  cmap=plt.cm.RdBu, vmin=-scale/2, vmax=scale/2);
    p = plt.axis('off')


#### Compare with the instructor's global model

Your instructor trained a logistic regression on a different slice of Quick Draw (indices 1000-1499, 500 samples per class). Let's see how it performs on YOUR drawing.


In [ ]:
url = f'http://{SERVER_IP}:{HTTP_PORT}/global_model.npy'
urllib.request.urlretrieve(url, 'global_model.npy')
w = np.load('global_model.npy', allow_pickle=True).item()
print('Global model weights downloaded.')


In [ ]:
global_model = LogisticRegression(max_iter=1, warm_start=True, solver='saga')
dummy_X = np.zeros((nclasses * 2, 784))
dummy_y = np.repeat(np.arange(nclasses), 2)
global_model.fit(dummy_X, dummy_y)
global_model.coef_ = w['coef']
global_model.intercept_ = w['intercept']
print('Global model loaded.')


In [ ]:
test_probs = global_model.predict_proba(test_sample)

sns.barplot(x=CLASSES, y=test_probs.squeeze());
plt.ylabel('Probability');
plt.xlabel('Class');

test_pred = global_model.predict(test_sample)
print('Global model predicts:', CLASSES[test_pred[0]])


#### Federated Learning

Now you will participate in Federated Learning (FL) rounds. The server sends you a starting model, you fine-tune it on YOUR local Quick Draw data, and send back only the updated weights (your raw data never leaves your computer).

The server averages the weights from all participants (FedAvg) and sends the improved model back. We'll try two setups:
- **Setup 2:** only Student A participates (1 client)
- **Setup 3:** both students participate (2 clients)


In [ ]:
class SketchClient(fl.client.NumPyClient):
    def __init__(self, X_train, y_train, X_val, y_val):
        self.X_train = X_train
        self.y_train = y_train
        self.X_val = X_val
        self.y_val = y_val
        self.model = LogisticRegression(
            penalty='l2', max_iter=1, warm_start=True, solver='saga',
        )
        dummy_X = np.zeros((nclasses * 2, 784))
        dummy_y = np.repeat(np.arange(nclasses), 2)
        self.model.fit(dummy_X, dummy_y)

    def get_parameters(self, config):
        return [self.model.coef_, self.model.intercept_]

    def fit(self, parameters, config):
        self.model.coef_ = parameters[0]
        self.model.intercept_ = parameters[1]
        self.model.fit(self.X_train, self.y_train)
        return [self.model.coef_, self.model.intercept_], len(self.X_train), {}

    def evaluate(self, parameters, config):
        self.model.coef_ = parameters[0]
        self.model.intercept_ = parameters[1]
        loss = float(log_loss(self.y_val, self.model.predict_proba(self.X_val)))
        acc = float(self.model.score(self.X_val, self.y_val))
        return loss, len(self.X_val), {'accuracy': acc}

print('SketchClient ready.')


#### Setup 2 — Only Student A participates

**Student A:** Run the cell below. It connects to the server for 3 FL rounds.
**Student B:** Do NOT run the cell below — wait for Setup 3.


In [ ]:
print('Connecting to', SERVER_ADDRESS, 'for Setup 2 (1 client)...')

client = SketchClient(X_train_scaled, y_train, X_test_scaled, y_test)
fl.client.start_numpy_client(server_address=SERVER_ADDRESS, client=client)

fl_model_s2 = client.model
print('Setup 2 complete.  Local val accuracy:', fl_model_s2.score(X_test_scaled, y_test))


#### Both students: download Setup 2 results

Download the final aggregated model from round 3 and test it on your drawing.
(Download BEFORE the instructor restarts the server for Setup 3.)


In [ ]:
url = f'http://{SERVER_IP}:{HTTP_PORT}/aggregated_round_3.npy'
urllib.request.urlretrieve(url, 'agg_round_3.npy')
w = np.load('agg_round_3.npy', allow_pickle=True).item()

fl_model_s2 = LogisticRegression(max_iter=1, warm_start=True, solver='saga')
dummy_X = np.zeros((nclasses * 2, 784))
dummy_y = np.repeat(np.arange(nclasses), 2)
fl_model_s2.fit(dummy_X, dummy_y)
fl_model_s2.coef_ = w['coef']
fl_model_s2.intercept_ = w['intercept']

test_probs = fl_model_s2.predict_proba(test_sample)
sns.barplot(x=CLASSES, y=test_probs.squeeze());
plt.ylabel('Probability');
plt.xlabel('Class');

test_pred = fl_model_s2.predict(test_sample)
print('Setup 2 predicts:', CLASSES[test_pred[0]])


#### Setup 3 — Both students participate

Now the server waits for **2 clients**. Both students run the cell below at approximately the same time.


In [ ]:
print('Connecting to', SERVER_ADDRESS, 'for Setup 3 (2 clients)...')

client = SketchClient(X_train_scaled, y_train, X_test_scaled, y_test)
fl.client.start_numpy_client(server_address=SERVER_ADDRESS, client=client)

fl_model_s3 = client.model
print('Setup 3 complete.  Local val accuracy:', fl_model_s3.score(X_test_scaled, y_test))


#### Both students: download Setup 3 results

Download the final aggregated model and test it on your drawing.


In [ ]:
url = f'http://{SERVER_IP}:{HTTP_PORT}/aggregated_round_3.npy'
urllib.request.urlretrieve(url, 'agg_round_3_v2.npy')
w = np.load('agg_round_3_v2.npy', allow_pickle=True).item()

fl_model_s3 = LogisticRegression(max_iter=1, warm_start=True, solver='saga')
dummy_X = np.zeros((nclasses * 2, 784))
dummy_y = np.repeat(np.arange(nclasses), 2)
fl_model_s3.fit(dummy_X, dummy_y)
fl_model_s3.coef_ = w['coef']
fl_model_s3.intercept_ = w['intercept']

test_probs = fl_model_s3.predict_proba(test_sample)
sns.barplot(x=CLASSES, y=test_probs.squeeze());
plt.ylabel('Probability');
plt.xlabel('Class');

test_pred = fl_model_s3.predict(test_sample)
print('Setup 3 predicts:', CLASSES[test_pred[0]])


#### Compare all models


In [ ]:
models = [
    ('Your model (Setup 1)', clf),
    ('Global model', global_model),
    ('Setup 2 - FL (just you)', fl_model_s2),
    ('Setup 3 - FL (both)', fl_model_s3),
]

print(f'{"Model":<30} {"Prediction":<12} {"Confidence":<10}')
print('-' * 55)
for name, m in models:
    pred = m.predict(test_sample)[0]
    prob = m.predict_proba(test_sample)[0][pred]
    print(f'{name:<30} {CLASSES[pred]:<12} {prob:.1%}')


#### Done!

| Concept | What happened |
|---------|--------------|
| **Local model** | Trained only on your Quick Draw slice |
| **Global model** | Pre-trained on different data (500/class) |
| **Federated Learning** | Model improved by averaging weights across participants |
| **Privacy** | Your drawing never left your computer - only weights were shared |
